In [58]:
%load_ext autoreload
%autoreload 2
import torch 
import torchvision
from torchvision import transforms
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt
from physics_utils import *
from dataset_dataloader import *
import yaml


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [50]:
#temp_train_ds=DatasetMaker('../data/train_data.csv',transforms=transforms.Compose([FftTransform(),transforms.ToTensor()]))
#all=[temp_train_ds[i][0] for i in range(len(temp_train_ds))]
#a=torch.stack(all,dim=0)
#train_mean=a[:,0,:,:].mean().item()
#train_std=a[:,0,:,:].std().item()  ## add these stats to the config file for further use

In [51]:
config_path = '../src/config.yaml'
with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

In [ ]:
config['stats']['train_mean'] = round(train_mean, 4)
config['stats']['train_std'] = round(train_std, 4)
with open(config_path, 'w') as f:
    yaml.dump(config, f, default_flow_style=False)
print("YAML updated")

YAML updated


In [52]:
train_transform = transforms.Compose([FftTransform(width=config['fft_params']['notch_width'], notch_depth=config['fft_params']['notch_depth'],
                                                   apply_bilateral=config['fft_params']['apply_bilateral']),
                                      transforms.RandomHorizontalFlip(p=0.5),
                                      transforms.RandomVerticalFlip(p=0.5),
                    # rotations (90 deg increase) to not destroy the fft transform
                                      transforms.RandomChoice([
                                      transforms.RandomRotation((0, 0)),
                                      transforms.RandomRotation((90, 90)),
                                      transforms.RandomRotation((180, 180)),
                                      transforms.RandomRotation((270, 270))]),
                                      transforms.ToTensor(),
                                      transforms.Normalize(mean=[config['stats']['train_mean']], std=[config['stats']['train_std']])])

val_transform =transforms.Compose([FftTransform(width=config['fft_params']['notch_width'], notch_depth=config['fft_params']['notch_depth'],
                                                    apply_bilateral=config['fft_params']['apply_bilateral']),
                                   transforms.ToTensor(),
                                   transforms.Normalize(mean=[config['stats']['train_mean']], std=[config['stats']['train_std']])])

In [53]:
train_ds=DatasetMaker(data_csv_path=config['data_path']['train'],transforms=train_transform)
val_ds=DatasetMaker(data_csv_path=config['data_path']['val'],transforms=val_transform)
test_ds=DatasetMaker(data_csv_path=config['data_path']['test'],transforms=val_transform)